<a href="https://colab.research.google.com/github/eshikajindal24/UCS547_Accelerated_data_science/blob/main/UCS547_102303954_Assign2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT 2**

1. Identify !, %, and %% used in cell in Google Colab.

->   ! (Shell command)-Used to run linux terminal commands from a notebook cell.% (Line magic)-Used for single-line magic commands.
%% (Cell magic)-Used to apply magic command to the entire cell.

2. Identify all key nvidia-smi commands with multiple options.

-> nvidia-smi = NVIDIA System Management Interface
-It is used to monitor GPU usage, memory and processes.
*  nvidia-smi -L: List GPUs
*  nvidia-smi -q: Detailed GPU info
*  nvidia-smi -q -d MEMORY: Memory details
*  nvidia-smi -q -d UTILIZATION : GPU utilization
*  nvidia-smi pmon : Process monitoring
*  nvidia-smi -l 1 : Refresh every 1 second
*  nvidia-smi --help : All options

3. Debug common CUDA errors (zero output, incorrect indexing, PTX errors).

-> (a) ***Zero Output***:
**Causes**:
Printing from CPU instead of GPU
**Fix**:
kernel<<<1,8>>>();
cudaDeviceSynchronize();

(b) **Incorrect Indexing**:
**Causes**:
Wrong global thread ID calculation
**Fix**:
int id = blockIdx.x * blockDim.x + threadIdx.x;

(c) **PTX Errors**
**Causes**:
-NVCC version mismatch
-Unsupported GPU architecture
**Fix**:
nvcc -arch=sm_75 program.cu

4. Write a CUDA C/C++ program to demonstrate GPU kernel execution and thread indexing.
a. Launch a CUDA kernel using: 1 block and 8 threads
b. Each thread must print: Hello from GPU thread <global_thread_id>
c. Compute the global thread ID using: global_thread_id = blockIdx.x * blockDim.x +
threadIdx.x
d. Clearly separate: Host code (CPU) & Device code (GPU kernel)

In [1]:
!nvidia-smi

Sun Feb 15 16:22:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%writefile ques4.cu
#include <stdio.h>

__global__ void ques4() {
    int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from GPU thread %d\n", global_thread_id);
}

int main() {
    printf("Hello from CPU (Host Code)\n");

    ques4<<<1,8>>>();
    cudaDeviceSynchronize();

    return 0;
}


Overwriting ques4.cu


In [4]:
!nvcc ques4.cu -o ques4

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [5]:
!./ques4

Hello from CPU (Host Code)
Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7


5. Write a CUDA program to demonstrate host and device memory separation.
a. Create an integer array of size 5 on the host (CPU).
b. Allocate corresponding memory on the device (GPU) using cudaMalloc().
c. Copy data from host to device using cudaMemcpy().
d. Launch a kernel where GPU threads print values from device memory.
e. Copy the data back from device to host and print it on CPU.

In [6]:
%%writefile ques5.cu
#include <stdio.h>

__global__ void ques5(int *d_arr)
{
    int idx = threadIdx.x;
    printf("GPU: Value at index %d: %d\n", idx, d_arr[idx]);
}

int main()
{
    int h_arr[5] = {10, 20, 30, 40, 50};
    int *d_arr;

    printf("CPU: Values before copy:\n");
    for (int i = 0; i < 5; i++)
    {
        printf("%d ", h_arr[i]);
    }
    printf("\n\n");

    cudaMalloc((void **)&d_arr, 5 * sizeof(int));

    cudaMemcpy(d_arr, h_arr, 5 * sizeof(int), cudaMemcpyHostToDevice);

    ques5<<<1, 5>>>(d_arr);

    cudaDeviceSynchronize();

    cudaMemcpy(h_arr, d_arr, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nCPU: Data copied from CPU memory to GPU memory:\n");
    for (int i = 0; i < 5; i++)
    {
        printf("%d ", h_arr[i]);
    }
    printf("\n");

    cudaFree(d_arr);

    return 0;
}


Writing ques5.cu


In [7]:
!nvcc ques5.cu -o abc

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [8]:
!./abc

CPU: Values before copy:
10 20 30 40 50 

GPU: Value at index 0: 10
GPU: Value at index 1: 20
GPU: Value at index 2: 30
GPU: Value at index 3: 40
GPU: Value at index 4: 50

CPU: Data copied from CPU memory to GPU memory:
10 20 30 40 50 


6. Compare CPU times of List/tuple with Numpy arrays.

In [11]:
import time, numpy as np
n = 10**6

l1 = list(range(n));
l2 = list(range(n))
t = time.time(); [l1[i]+l2[i] for i in range(n)]
print("List:", time.time()-t)

t1 = tuple(range(n));
t2 = tuple(range(n));
t = time.time();
tuple(t1[i]+t2[i] for i in range(n))
print("Tuple:", time.time()-t)

a = np.arange(n);
b = np.arange(n);
t = time.time();
a+b
print("NumPy:", time.time()-t)


List: 0.09419822692871094
Tuple: 0.09905195236206055
NumPy: 0.0017910003662109375
